# Scientific Programming with Python 
## (Winter 2025/26)

# Session 08: Pandas III

* Finish Pandas
* Some more python: Widgets in notebooks

# Recap

Last week we saw:
* Pandas Series (one-dimensional ndarray with axis labels)
  - creating series
  - indexing, aggregation,
* Pandas DataFrames (two-dimensional, size-mutable, potentially heterogeneous tabular data)
  - construction of DataFrames
  - access `[]`, index based (`loc`) and numerical (`iloc`), slicing
* Pandas Indices (Convenient way to access elements)
  - changing indices for Series and DataFrames
* Boolean indexing

# Ufuncs and Aggregation

## Aggregation in Pandas

Aggregations are functions, where one or more dimensions of data are collapsed onto a single value, like the `max`, `sum` or `mean`- functions.

Stat-operations generally *exclude* missing data.

### For Series

In [ ]:
import pandas as pd
import numpy as np

a = np.arange(7)
ser = pd.Series(a**2, index=a)
ser

In [ ]:
ser.sum()
#mean(), median(), min(), max(), ...

### For DataFrames

In [ ]:
df = pd.DataFrame({'A': a**2,
                   'B': a**3})
df

In [ ]:
df.mean()

In [ ]:
df.mean(axis=0)

In [ ]:
df.mean(axis='rows')

In [ ]:
df.mean(axis=1)

The following table summarizes some other built-in Pandas aggregations:

| Aggregation              | Description                     |
|--------------------------|---------------------------------|
| ``count()``              | Total number of items           |
| ``first()``, ``last()``  | First and last item             |
| ``mean()``, ``median()`` | Mean and median                 |
| ``min()``, ``max()``     | Minimum and maximum             |
| ``std()``, ``var()``     | Standard deviation and variance |
| ``mad()``                | Mean absolute deviation         |
| ``prod()``               | Product of all items            |
| ``sum()``                | Sum of all items                |

These are all methods of ``DataFrame`` and ``Series`` objects.

## Ufuncs


We know Ufuncs (universial functions) already from Numpy: It are vectorized functions that change all values of an array simultaneously. 

Pandas does the same, with a nice twist: for unary operations like negation and trigonometric functions, these ufuncs will *preserve index and column labels* in the output, and for binary operations such as addition and multiplication, Pandas will automatically *align indices* when passing the objects to the ufunc.


This means that keeping the context of data and combining data from different sources –both potentially error-prone tasks with raw NumPy arrays– become essentially foolproof ones with Pandas.

In [ ]:
rng = np.random.RandomState(0)
df = pd.DataFrame(rng.randint(0, 10, (3, 4)),
                  columns=['A', 'B', 'C', 'D'])
df

In [ ]:
np.exp(df)

### UFuncs: Index Alignment

For binary operations on two ``Series`` or ``DataFrame`` objects, Pandas will align indices in the process of performing the operation.
This is very convenient when working with incomplete data.

In [ ]:
area = pd.Series({'Alaska': 1723337, 'Texas': 695662,
                  'California': 423967}, name='area')
population = pd.Series({'California': 38332521, 'Texas': 26448193,
                        'New York': 19651127}, name='population')
area

In [ ]:
population

In [ ]:
area / population

In [ ]:
"divide" in dir(pd.DataFrame)

In [ ]:
popdens = area.divide(population, fill_value=0)
popdens

In [ ]:
popdens = popdens.replace([np.inf, -np.inf], np.nan)
popdens.dropna()

In [ ]:
A = pd.DataFrame(rng.randint(0, 20, (2, 2)),
                 columns=list('AB'))
A

In [ ]:
B = pd.DataFrame(rng.randint(0, 20, (3, 3)),
                 columns=list('ABC'))
B

In [ ]:
A+B

In [ ]:
A.add(B, fill_value=0)

### More Index-Alignment

In [ ]:
df = pd.DataFrame({'a': np.random.randint(3, size=10)}, index=np.arange(1, 20, 2))
df

Let's add a new column to this DataFrame!

In [ ]:
tmp = pd.Series([0]*len(df.index))
tmp

In [ ]:
#df['new'] = tmp   #changes the original one
df.assign(new=tmp) #creates a copy

In [ ]:
old_aligned, new_aligned = df.align(tmp, axis=0)
old_aligned

In [ ]:
new_aligned

In [ ]:
old_aligned.assign(new=new_aligned)

In [ ]:
tmp = pd.Series([0]*len(df.index), index=df.index)
tmp

In [ ]:
df['new'] = tmp
df

## .agg()

If you want to apply more than one operation (ufunc/aggregation), use `.agg()`:

In [ ]:
df = pd.DataFrame([[1, 2, 3],
                   [4, 5, 6],
                   [7, 8, 9],
                   [np.nan, np.nan, np.nan]],
                  columns=['A', 'B', 'C'])
df

In [ ]:
df.agg(['sum', 'min'])

In [ ]:
#you can also use different aggregations for different columns: 
df.agg({'A' : ['sum', 'min'], 'B' : ['min', 'max']})

## apply()

While some ufuncs (like cumsum or exp) are pre-defined by pandas, the method `apply` can be used to run an arbitrary function on all elements of a Series or DataFrame.

In [ ]:
a = np.arange(7)
df = pd.DataFrame({'A': a**2,
                   'B': a**3})
df

In [ ]:
df.cumsum()

In [ ]:
df.apply(np.cumsum)

In [ ]:
df["A_cumsum"] = df.cumsum()["A"]
df["B_cumsum"] = df.apply(np.cumsum)["B"]
df

Using Lambda-functions, we can combine `apply` with arbitrary functions. Note that the argument of the function is always an entire column of the dataset.

In [ ]:
df.apply(lambda x: print(x, end='\n\n'))

In [ ]:
df

In [ ]:
df['A'] + 1

In [ ]:
df.apply(lambda x: x+1)

In [ ]:
def my_more_complex_func(ser):
    res = []
    for elem in ser:
        res.append(elem if elem > 16 else -elem)
    return res

In [ ]:
df.apply(my_more_complex_func)

In [ ]:
df

In [ ]:
df.apply(lambda x: x.max() - x.min())

Note that `apply` works for both DataFrames and Series!

In [ ]:
df["A"].apply(lambda x: print(x))

In [ ]:
df["A_normed"] = df["A"].apply(lambda x: x/df["A"].max())
df

We can even use dictionaries with the apply-function!

In [ ]:
z_moves = {"Normal": "Breakneck Blitz", "Fighting": "All-Out Pummeling", "Flying": "Supersonic Skystrike", "Poison": "Acid Downpour", "Ground": "Tectonic Rage", "Rock": "Continental Crush", "Bug": "Savage Spin-Out", "Ghost": "Never-Ending Nightmare",
"Steel": "Corkscrew Crash", "Fire": "Inferno Overdrive", "Water": "Hydro Vortex", "Grass": "Bloom Doom", "Electric": "Gigavolt Havoc", "Psychic": "Shattered Psyche", "Ice": "Subzero Slammer", "Dragon": "Devastating Drake", "Dark": "Black Hole Eclipse", "Fairy": "Twinkle Tackle"}
df = pd.read_csv("Pokemon.csv")
df.head()

In [ ]:
df["Z-Move"] = df["Type 1"].apply(lambda x:z_moves[x])
df.head()

Using `apply`, we can also convert a list of Series into a DataFrame, by making the individual columns to Series:

In [ ]:
s = pd.Series([ ['Red', 'Green', 'White'], ['Red', 'Black'], ['Yellow']]) 
print(type(s))
s

In [ ]:
pd.Series([1, 2, 3])

In [ ]:
df = s.apply(pd.Series)
print(type(df))
df

**Note on speed:**

According to ([1]), `apply()` is twice as fast as looping through a df's `iterrows()`, and 8 times as fast as looping over python lists.

Note however, that while `apply()` is much faster at looping over the rows of your DataFrame/Series (by taking advantage of a number of internal optimizations, such as using iterators in Cython), it still inherently loops through rows. Whatever you're applying, you're still executing it once for every row. So, wherever you can use make use of vectorized Ufuncs, do so - that is far more optimized and parallelized - for ([1]) exchanging the haversine distance formula with it's vectorized counterpart led to a 50-fold-decrease in time!

\(1): https://engineering.upside.com/a-beginners-guide-to-optimizing-pandas-code-for-speed-c09ef2c6a4d6


[1]: https://engineering.upside.com/a-beginners-guide-to-optimizing-pandas-code-for-speed-c09ef2c6a4d6 

# Group-By

## Split-Apply-Combine

While simple operations are already pre-defined by pandas, custom aggregations and operations can be performed via **group-by**. The group-by operation can be described as having the following steps:

* **Splitting** the data into groups based on some criteria (breaking up and grouping depending on the value of a key)
* **Applying** a function to each group independently (aggregation, transformation, filtering, ...)
* **Combining** the results into a data structure

A typical example, for where the *apply* is a summerization aggregation, is illustrated here:

![](split-apply-combine.png)

In [ ]:
tmp = np.array([list("ABCABC"), np.arange(1,7)]).T
tmp

In [ ]:
df = pd.DataFrame(tmp, columns=["key", "data"])
df["data"] = pd.to_numeric(df["data"])
df

In [ ]:
df.groupby("key")

Note that what is returned is not a set of `DataFrames`, but a `DataFrameGroupBy` object. This object is where the magic is: you can think of it as a special view of the `DataFrames`, which is poised to dig into the groups but does no actual computation until the aggregation is applied. This "lazy evaluation" approach means that common aggregates can be implemented very efficiently in a way that is almost transparent to the user.

To produce a result, we can apply an aggregate to this `DataFrameGroupBy` object, which will perform the appropriate apply/combine steps to produce the desired result:

In [ ]:
df.groupby("key").sum().reset_index()

In [ ]:
df.groupby("key")["data"].sum()
# we can do column indexing just like on a normal DataFrame

### Iteration over groups

The ``GroupBy`` object supports direct iteration over the groups, returning each group as a ``Series`` or ``DataFrame``:

In [ ]:
df

In [ ]:
for (key, _) in df.groupby("key"):
    print(key)

In [ ]:
for (_, group) in df.groupby("key"):
    print(group, "\n")

In [ ]:
pkm = pd.read_csv('Pokemon.csv')
pkm.groupby('Generation')['Total'].mean()

### Dispatch methods

Any method not explicitly implemented by the ``GroupBy`` object will be passed through and called on the groups, whether they are ``DataFrame`` or ``Series`` objects.
For example, you can use the ``describe()`` method of ``DataFrame``s to perform a set of aggregations that describe each group in the data:

In [ ]:
df.describe()

In [ ]:
df.groupby("key").describe()

In [ ]:
df = pd.read_csv("Pokemon_no_duplicates.csv", index_col=0)
df.groupby('Generation')["Name"].nunique()

# IPython Widgets

The `ipywidgets` package allow to display widgets in Jupyter notebooks.

```
conda install -c conda-forge ipywidgets ipympl matplotlib
```

You may have to restart jupyter lab (not just the current kernel) for the installation to take effect.

In [ ]:
from ipywidgets import interact, IntSlider

interact(lambda x: x, x=IntSlider());

### Widgets

`ipywdigets` provides sdever widgets that can be used to input values

In [ ]:
from ipywidgets import Text, IntSlider, Dropdown

# Text-input
tb1 = Text(value='some text', 
                   description='Text-Widget: ')
display(tb1)

In [ ]:
# IntSlider-Widget
is1 = IntSlider(value=7, min=0, max=10, 
                        step=1, description='IntSlider: ')
display(is1)

In [ ]:
# Dropdown-Widget
dd1 = Dropdown(options=['1', '2', '3'], 
                       value='2', description='Dropdown: ')
display(dd1)

#### Widget values

The value of a widget ist accessible via the `.value` property:

In [ ]:
print(f'Textbox tb1 has the value "{tb1.value}"')

It is also possible to change that value:

In [ ]:
tb1.value = "hello"

### Event handler

* events happen when the user interacts with a widget
* event handlers are functions to be called when in case of an event

In [ ]:
import ipywidgets
from ipywidgets import Text, Button, Layout

# create two text widgets
textbox1 = Text(description='a'); display(textbox1)
textbox2 = Text(description='b'); display(textbox2)
# ... and a Button
button = Button(description='Compute!', 
                layout=Layout(width='200px'))
button.style.button_color = 'lightgreen'
display(button)

# ... and an output fielde
textbox3 = Text(description='Sum')
display(textbox3)

In [ ]:
# define an event handler for the button
def on_button_clicked(sender):
    a = int(textbox1.value)
    b = int(textbox2.value)
    textbox3.value = str(a+b)

# ... and connect it to the "on_click" event
button.on_click(on_button_clicked)

## The `interact` mechanism

The `interact` functions

In [ ]:
def myfunc1(x):  
     print('Selection: ' + str(x)) 

# interact creates a Dropdown widget 
interact(myfunc1, x = [1, 2, 3]); 

In [ ]:
def myfunc2(x):  
     print(f'Your input: "{x}"')

# interact creates a text box
interact(myfunc2, x = 'Input some text'); 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def myplot(xmax):
    x = np.arange(0, xmax, 0.01)
    y = np.sin(2*np.pi*x)
    plt.plot(x,y)
    
interact(myplot, xmax = (np.pi/2,4*np.pi));

#### The function `interactive_output`

The `interactive_output` allows more control to align the graphical elements:

In [ ]:
import ipywidgets
from ipywidgets import FloatSlider, VBox, Layout
from ipywidgets import interactive_output

def myplot(xmax):
    x = np.arange(0, xmax, 0.01)
    y = np.sin(2*np.pi*x)
    plt.plot(x,y)

ui_xmax = FloatSlider(description='xmax', value=10, min=np.pi/2, max=2*np.pi);
out = interactive_output(myplot, {'xmax': ui_xmax})
display(VBox([ui_xmax,out], layout=Layout(width='50%',border='1px dotted blue')))

### Partial update of matplotlib figures

* `plt.plot()` recreates the full figure
* updating only the data may be more efficient

In [ ]:
%matplotlib widget
from ipywidgets import interact, FloatSlider
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(0, 2*np.pi, 500)
fig, ax = plt.subplots()
line, = ax.plot(x, np.sin(x))
ax.set_ylim(-2, 2)

def update(freq=1.0, amp=1.0, phase=0.0):
    y = amp * np.sin(freq * x + phase)
    line.set_ydata(y)
    fig.canvas.draw_idle()  # request a redraw

In [ ]:
interact(
    update,
    freq=FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1),
    amp=FloatSlider(value=1.0, min=0.0, max=2.0, step=0.1),
    phase=FloatSlider(value=0.0, min=-np.pi, max=np.pi, step=0.1)
)

In [ ]:
import ipywidgets as widgets
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(0, 2*np.pi, 500)
fig, ax = plt.subplots()
line, = ax.plot(x, np.sin(x))
ax.set_ylim(-2, 2)

freq = widgets.FloatSlider(description='freq', value=1.0, min=0.1, max=5.0, step=0.1)
amp = widgets.FloatSlider(description='amp', value=1.0, min=0.0, max=2.0, step=0.1)
phase = widgets.FloatSlider(description='phase', value=0.0, min=-np.pi, max=np.pi, step=0.1)

def on_change(change):
    y = amp.value * np.sin(freq.value * x + phase.value)
    line.set_ydata(y)
    fig.canvas.draw_idle()

for w in (freq, amp, phase):
    w.observe(on_change, 'value')

widgets.VBox([freq, amp, phase])

### More examples

In [ ]:
import pandas as pd

def display_dataframe(df, rows=6, cols=6):
    with pd.option_context('display.max_rows', rows, 
                    'display.max_columns', cols):  
        display(df)

def select_data(columns): 
    df2 = df.loc[:,columns];  
    ax = df2.plot(); 
    ax.set_ylabel('Dayly consumption (GWh)');

In [ ]:
import ipywidgets as widgets

out1, out2 = widgets.Output(), widgets.Output()
with out1:
    file = 'opsd_2018.csv'
    df = pd.read_csv(file, header=0, sep = ",", index_col=0, parse_dates=True) 
    display_dataframe(df)
with out2: 
    ui_spalten = widgets.Dropdown(description='Selection:', 
                                  options=df.columns[0:], value='Verbrauch', 
                                  layout=widgets.Layout(width='auto'))
    out = widgets.interactive_output(select_data, {'columns': ui_spalten}); 
    display(widgets.VBox([ui_spalten, out]))
    
tab = widgets.Tab(children = [out1, out2], layout=widgets.Layout(width='70%'))
tab.set_title(0, 'Data')
tab.set_title(1, 'Figure')
display(tab)